In [ ]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")
library("parallel")
dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }

dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }

num_cores <- detectCores()
register(MulticoreParam(num_cores))
bpparam <- MulticoreParam(num_cores, log = TRUE, stop.on.error = FALSE)


# Current Version

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/allele_only/THP1MacrophageBrainR1R2merged20240404_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/THP1MacrophageBrainR1R2merged20240404_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_THP1Macrophage_barcodesBrainR1R2merged20240404.csv", header=TRUE,row.names = 1)

df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele+Parent,
                          rnaDesign = ~ Tissue+Animal+Allele_String+Brain_interaction,
                          reducedDesign = ~ Tissue+Animal+Allele_String,
                          #mode="scale"
                          )
                
res <-testLrt(obj)
write.csv(res,"20240813_comparative_THP1PseudobarcodesvsBrainR1R2merged_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-08-14 12:56:43.857584
Success: TRUE

Task duration:
   user  system elapsed 
 47.428   1.075  50.426 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6172866 329.7   11607064 619.9  9457893 505.2
Vcells 11618843  88.7   19192058 146.5 19192058 146.5

Log messages:
INFO [2024-08-14 12:55:53] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-08-14 12:57:00.140826
Success: TRUE

Task duration:
   user  system elapsed 
 64.263   0.823  67.616 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6174143 329.8   11607064 619.9  9457893 505.2
Vcells 11623391  88.7   19192058 146.5 19192058 146.5

Log messages:
INFO [2024-08-14 12:55:52] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 9
Node: 16
Timestamp: 2024-08-14 12:5

In [ ]:
###################Whole Brain R1R2merged###################
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes_reshaped.csv", header=TRUE)

df_DNA <-as.matrix(dropEnhancer(df_DNA))
df_RNA <-as.matrix(dropEnhancer(df_RNA))

df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]

annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_barcodes.csv")

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Tissue+Animal+Allele_String,
                          reducedDesign = ~ Tissue+Animal,
                          #mode="scale"
                          )
                  
res <-testLrt(obj)
write.csv(res,"20240409_comparative_BrainR1R2merged20240404_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-04-09 15:55:47.780437
Success: TRUE

Task duration:
   user  system elapsed 
 11.864   0.294  21.460 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6283518 335.6   11969565 639.3  8889702 474.8
Vcells 11463326  87.5   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:55:26] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 23
Node: 2
Timestamp: 2024-04-09 15:55:54.29804
Success: TRUE

Task duration:
   user  system elapsed 
 18.013   0.598  28.007 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6284862 335.7   11969565 639.3  8889702 474.8
Vcells 11468021  87.5   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:55:26] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 15
Node: 10
Timestamp: 2024-04-09 15:5

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Cortex_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Cortex_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Cortex_barcodes.csv")
df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )             
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          #mode="scale"
                          )                                       
res <-testLrt(obj)
write.csv(res,"20240409_comparative_BrainR1R2merged20240404_Cortex_allele.csv")

df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Striatum_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Striatum_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Striatum_barcodes.csv")
df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )          
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          #mode="scale"
                          )                                   
res <-testLrt(obj)
write.csv(res,"20240409_comparative_BrainR1R2merged20240404_Striatum_allele.csv")


df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Hippocampus_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_Hippocampus_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Hippocampus_barcodes.csv")
df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]
#Match row names
annot_DNA<-dropX(annot_DNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}
#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))
#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                  control=control,
                  BPPARAM = bpparam
                  )
              
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          #mode="scale"
                          )                         
res <-testLrt(obj)
write.csv(res,"20240409_comparative_BrainR1R2merged20240404_Hippocampus_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-04-09 15:57:01.021238
Success: TRUE

Task duration:
   user  system elapsed 
  8.023   0.343  17.881 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295168 336.2   11969565 639.3  8889702 474.8
Vcells 11334214  86.5   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:56:43] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 16
Node: 9
Timestamp: 2024-04-09 15:57:02.433407
Success: TRUE

Task duration:
   user  system elapsed 
 12.309   0.559  18.701 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295995 336.3   11969565 639.3  8889702 474.8
Vcells 11339522  86.6   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:56:42] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 12
Node: 13
Timestamp: 2024-04-09 15:

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_StriatumvsCortex_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_StriatumvsCortex_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_StriatumvsCortex_barcodes.csv")
df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                 control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Tissue+Animal+Allele_String+Striatum_interaction,
                          reducedDesign = ~ Tissue+Animal+Allele_String,
                          )
                        
res <-testLrt(obj)
write.csv(res,"20240409_comparative_StriatumvsCortex_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-04-09 15:59:47.867964
Success: TRUE

Task duration:
   user  system elapsed 
 12.029   0.280  12.549 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295213 336.3   11969565 639.3  8889702 474.8
Vcells 11412732  87.1   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:59:35] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 7
Node: 18
Timestamp: 2024-04-09 15:59:53.024422
Success: TRUE

Task duration:
   user  system elapsed 
 16.904   0.255  18.291 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295605 336.3   11969565 639.3  8889702 474.8
Vcells 11415352  87.1   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 15:59:34] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 18
Node: 7
Timestamp: 2024-04-09 15:5

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_HippocampusvsCortex_DNA_matched_barcodes_reshaped.csv", header=TRUE)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_HippocampusvsCortex_RNA_matched_barcodes_reshaped.csv", header=TRUE)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_HippocampusvsCortex_barcodes.csv")
df_DNA<-df_DNA[2:nrow(df_DNA), ]
df_RNA<-df_RNA[2:nrow(df_RNA), ]

#Match row names
annot_DNA<-dropX(annot_DNA)

#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Read negative controls
negative <- read.csv('indexing/mpra3_negatives.csv' , sep="\t",header=FALSE)
control <- as.character(negative$V1)#V1 is the sequence_ID
df_RNA <-as.matrix(dropEnhancer(df_RNA))
df_DNA <-as.matrix(dropEnhancer(df_DNA))

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
#                 control=control,
                  BPPARAM = bpparam
                  )
                  
#enhancer level per tissue;
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Tissue+Animal+Allele_String+Hippocampus_interaction,
                          reducedDesign = ~ Tissue+Animal+Allele_String,
                          )
                  
res <-testLrt(obj)
write.csv(res,"20240409_comparative_HippocampusvsCortex_interaction.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-04-09 16:00:38.716182
Success: TRUE

Task duration:
   user  system elapsed 
 11.850   0.264  12.977 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295365 336.3   11969565 639.3  8889702 474.8
Vcells 11412815  87.1   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 16:00:25] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 5
Node: 20
Timestamp: 2024-04-09 16:00:43.460021
Success: TRUE

Task duration:
   user  system elapsed 
 17.096   0.275  18.357 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6295757 336.3   11969565 639.3  8889702 474.8
Vcells 11415435  87.1   19189761 146.5 19186821 146.4

Log messages:
INFO [2024-04-09 16:00:25] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 9
Node: 16
Timestamp: 2024-04-09 16:0

# Current Version, Allele Only

In [ ]:
###################Whole Brain R1R2merged###################
df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_barcodes.csv")
df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Tissue+Animal+Allele_String,
                          reducedDesign = ~ Tissue+Animal,
                          )
                                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_BrainR1R2merged20240404_allele.csv")



df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Cortex_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Cortex_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Cortex_barcodes.csv")
df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          )
                                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_BrainR1R2merged20240404_Cortex_allele.csv")



df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Striatum_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Striatum_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Striatum_barcodes.csv")
df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          )
                                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_BrainR1R2merged20240404_Striatum_allele.csv")



df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Hippocampus_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_Hippocampus_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_Hippocampus_barcodes.csv")
df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )
                  
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Animal+Allele_String,
                          reducedDesign = ~ Animal,
                          )
                                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_BrainR1R2merged20240404_Hippocampus_allele.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-16 19:11:01.15903
Success: TRUE

Task duration:
   user  system elapsed 
 10.985   0.240  12.026 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5708217 304.9    8993288 480.3  8408002 449.1
Vcells 10325636  78.8   17824054 136.0 17824051 136.0

Log messages:
INFO [2024-06-16 19:10:49] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 1
Node: 24
Timestamp: 2024-06-16 19:11:03.897368
Success: TRUE

Task duration:
   user  system elapsed 
 14.925   0.216  15.309 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5709271 305.0    8993288 480.3  8408002 449.1
Vcells 10329445  78.9   17824054 136.0 17824051 136.0

Log messages:
INFO [2024-06-16 19:10:48] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 19
Node: 6
Timestamp: 2024-06-16 19:11

# Motif-Allele

In [ ]:
df_DNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_DNA_matched_barcodes_motif.csv", header=TRUE,row.names = 1)
df_RNA <- read.csv("read_counts_R1R2/BrainR1R2merged20240404_RNA_matched_barcodes_motif.csv", header=TRUE,row.names = 1)
annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged_barcodes_motif.csv")
df_DNA <-as.matrix(df_DNA)
df_RNA <-as.matrix(df_RNA)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

#Create MPRA object
obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  BPPARAM = bpparam
                  )

#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   

obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele,
                          rnaDesign = ~ Tissue+Animal+Motif_Allele,
                          reducedDesign = ~ Tissue+Animal,
                          )
                                         
res <-testLrt(obj)
write.csv(res,"20240616_comparative_BrainR1R2merged20240404_motif.csv")

Fitting model...

############### LOG OUTPUT ###############
Task: 24
Node: 1
Timestamp: 2024-06-17 12:56:50.25431
Success: TRUE

Task duration:
   user  system elapsed 
  1.536   0.255   1.938 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5701754 304.6   10323991 551.4 10323991 551.4
Vcells 10210150  77.9   18611228 142.0 16161901 123.4

Log messages:
INFO [2024-06-17 12:56:48] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 10
Node: 15
Timestamp: 2024-06-17 12:56:52.890621
Success: TRUE

Task duration:
   user  system elapsed 
  4.640   0.311   5.162 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  5701710 304.6   10323991 551.4 10323991 551.4
Vcells 10210885  78.0   18611228 142.0 16161901 123.4

Log messages:
INFO [2024-06-17 12:56:47] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 8
Node: 17
Timestamp: 2024-06-17 12:5

# Interaction

In [ ]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")
library("parallel")
dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }

dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }

num_cores <- detectCores()
register(MulticoreParam(num_cores-1))
bpparam <- MulticoreParam(num_cores-1, log = TRUE, stop.on.error = FALSE)

df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_HippocampusvsCortex_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_HippocampusvsCortex_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_HippocampusvsCortex_barcodes.csv", header=TRUE,row.names=1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+Hippocampus_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                    
res <-testLrt(obj)              

write.csv(res,"20250519_comparative_BrainR1R2merged20240404_HippocampusvsCortex_interaction.csv") 

Fitting model...

############### LOG OUTPUT ###############
Task: 23
Node: 1
Timestamp: 2025-05-19 11:39:03.298075
Success: TRUE

Task duration:
   user  system elapsed 
 11.368   0.266  11.723 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6171454 329.6   11623774 620.8  8496602 453.8
Vcells 11137285  85.0   19180842 146.4 15606232 119.1

Log messages:
INFO [2025-05-19 11:38:51] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 17
Node: 7
Timestamp: 2025-05-19 11:39:12.793121
Success: TRUE

Task duration:
   user  system elapsed 
 20.973   0.279  21.435 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6171856 329.7   11623774 620.8  8496602 453.8
Vcells 11139769  85.0   19180842 146.4 15606232 119.1

Log messages:
INFO [2025-05-19 11:38:51] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 18
Node: 6
Timestamp: 2025-05-19 11:3

In [ ]:
#if (!require("BiocManager", quietly = TRUE))
    #install.packages("BiocManager")

#BiocManager::install("MPRAnalyze")
#Format the data into MPRA project
library("MPRAnalyze")
library("BiocParallel")
library("parallel")
dropEnhancer <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$enhancer_id	
    df <- df_RNA[ , !(names(df_RNA) %in% c("enhancer_id"))]
    return(df)
    }

dropX <- function(df_RNA){
    row.names(df_RNA) <-df_RNA$X
    df <- df_RNA[ , !(names(df_RNA) %in% c("X"))]
    return(df)
    }

num_cores <- detectCores()
register(MulticoreParam(num_cores-1))
bpparam <- MulticoreParam(num_cores-1, log = TRUE, stop.on.error = FALSE)

df_DNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_StriatumvsCortex_DNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)
df_RNA <- read.csv("read_counts_R1R2/allele_only/BrainR1R2merged20240404_StriatumvsCortex_RNA_matched_barcodes_reshaped_allele.csv", header=TRUE,row.names=1)

df_DNA<-as.matrix(df_DNA)
df_RNA<-as.matrix(df_RNA)

annot_DNA <-read.csv("annotation_barcodes/mpra3_annot_BrainR1R2merged20240404_StriatumvsCortex_barcodes.csv", header=TRUE,row.names=1)
#Format the columns as factor that is recognized by MPRAnalyze
for (i in colnames(annot_DNA)){
    annot_DNA[i] <- as.factor(annot_DNA[,i])
}

obj <- MpraObject(dnaCounts=df_DNA,
                  rnaCounts=df_RNA,
                  colAnnot=annot_DNA,
                  #control=control,
                  BPPARAM = bpparam
                  )
#Comparative analysisl; Tissue only
obj <- estimateDepthFactors(obj, lib.factor = c("Test"), which.lib = "both",depth.estimator = "uq")   
obj <- analyzeComparative(obj = obj, 
                          dnaDesign = ~ Barcode_Allele + Test,
                          rnaDesign = ~ Tissue+Allele_String+Striatum_interaction,
                          reducedDesign = ~ Tissue+Allele_String,
                          #mode="scale"
                          )                    
res <-testLrt(obj)              

write.csv(res,"20250519_comparative_BrainR1R2merged20240404_StriatumvsCortex_interaction.csv") 

Fitting model...

############### LOG OUTPUT ###############
Task: 23
Node: 1
Timestamp: 2025-05-19 11:44:32.453731
Success: TRUE

Task duration:
   user  system elapsed 
 11.756   0.305  12.272 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6175453 329.9   11623774 620.8  8496602 453.8
Vcells 11152748  85.1   19180842 146.4 15606232 119.1

Log messages:
INFO [2025-05-19 11:44:20] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 9
Node: 15
Timestamp: 2025-05-19 11:44:42.06794
Success: TRUE

Task duration:
   user  system elapsed 
 22.015   0.269  22.358 

Memory used:
           used  (Mb) gc trigger  (Mb) max used  (Mb)
Ncells  6175855 329.9   11623774 620.8  8496602 453.8
Vcells 11155232  85.2   19180842 146.4 15606232 119.1

Log messages:
INFO [2025-05-19 11:44:19] loading futile.logger package

stderr and stdout:


############### LOG OUTPUT ###############
Task: 2
Node: 22
Timestamp: 2025-05-19 11:44